# Signal Lab Quickstart

Notebook-first exploration of one concrete copy-trade signal: `sig_val_opp_flipper`.

This notebook keeps the important logic explicit so you can inspect:
- raw trades and train/val/test splits
- wallet metrics and copy-universe selection
- candidate BUY trades
- flipper cohort definition
- direct signal construction
- before/after trade quality after thresholding

Main stage1-style objects exposed here: `df_full`, `df_train`, `df_val`, `df_test`, `candidate_trades`, `c_train`, `c_val`, `c_test`.

In [16]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'signal_lab' else NOTEBOOK_DIR
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lib import (
    DEFAULT_TAGS,
    compute_copyable_notional,
    compute_opening_metrics,
    load_trades,
    split_data,
)
from polymarket_analysis.wallet_selection.volatility import compute_wallet_metrics
from signal_lab.filters import archetype_sets
from signal_lab.signal_engines import PositionSignalEngine, compute_hold_time_metrics
from signal_lab.signal_lib import (
    apply_rank_transformer,
    bootstrap_ic,
    compute_event_ic,
    evaluate_strategy,
    fit_rank_transformer,
    fit_roi_residualizer,
    residualized_roi,
)
from signal_lab.stage1 import evaluate_threshold_grid

from signal_lab.strategies import GamblerCapitulationSqueeze, FreshOppositeCrowdingFilter


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Parameters

In [17]:
COPY_MIN_BUY_ROI = 0.02
COPY_MIN_BUCKETS = 20
COPY_MIN_MARKETS = 15
COPY_MIN_TRADE_COUNT = 100
COPY_MAX_DD_TO_PNL = 0.6
COPY_MIN_COPYABLE_ROI = 0.05
ARCH_MIN_TRADE_COUNT = 100
COST_BPS = 0.0
THRESHOLD_MIN_TRADES = 20

ENGINE_COLS = [
    'wallet', 'condition_id', 'outcome', 'dt',
    'side', 'position', 'quantity', 'price',
]

SIGNAL_COL = 'sig_val_opp_flipper'
COPY_SIGNAL_COL = 'sig_copy_anti_crowding_flipper'
SCORE_COL = 'score_flipper'


## 1. Load raw trades and split them

In [18]:
df_full = load_trades(tags=DEFAULT_TAGS)
df_full = compute_copyable_notional(df_full)
df_train, df_val, df_test = split_data(df_full, method='chronological')

print(f'df_full={len(df_full):,} train={len(df_train):,} val={len(df_val):,} test={len(df_test):,}')


Markets: 1974837
Filtered markets for {'Weather'}: 91123
Loading 16 trade shards...
Total trades loaded: 14,250,603
Unique wallets: 4,082
Date range: 2025-01-09 15:32:39+00:00 -> 2026-07-27 06:12:25+00:00
Chronological split: train <= 2026-05-21T00:00:00Z, val <= 2026-06-23T00:00:00Z, test > 2026-06-23T00:00:00Z
Method: chronological  |  Unique end dates: 112  (train=44, val=33, test=35)

  Train:  4,337,961 trades  (15,923 markets)
  Val:    5,211,194 trades  (20,297 markets)
  Test:   4,701,448 trades  (20,655 markets)
  Total: 14,250,603 trades  (56,875 markets)
df_full=14,250,603 train=4,337,961 val=5,211,194 test=4,701,448


## 2. Compute wallet metrics directly in the notebook

In [19]:
wallet_metrics, _ = compute_wallet_metrics(df_train)
wallet_metrics['copyable_pnl_factor'] = np.clip(
    wallet_metrics['copyable_pnl'] / wallet_metrics['total_pnl'].replace(0, np.nan),
    0,
    1.0,
).fillna(0.0)
wallet_metrics['copyable_roi'] = wallet_metrics['average_roi'] * wallet_metrics['copyable_pnl_factor']

opening_metrics = compute_opening_metrics(df_train)
wallet_metrics = wallet_metrics.merge(opening_metrics, on='wallet', how='left')
for col in ['opening_roi', 'opening_pnl', 'opening_copyable_roi', 'opening_copyable_pnl']:
    wallet_metrics[col] = wallet_metrics[col].fillna(0.0)

hold_metrics = compute_hold_time_metrics(df_train)

display(wallet_metrics[['wallet', 'buy_roi', 'copyable_roi', 'trade_count', 'num_markets', 'num_buckets']].head())


,wallet,buy_roi,copyable_roi,trade_count,num_markets,num_buckets
0,0x0054ee7dfb882d2d016fa13ef5f5cdb3b0ebcf1f,0.008757,-0.000000,53,16,38
1,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0.039375,0.069611,210,180,208
2,0x00833cc2d777e6f2fc8437679124024ae6468cb1,-0.070000,0.000000,2,1,2
3,0x0141be702d272f17666e280303ad44e7bc0cc2da,0.028754,0.243228,79,13,34
4,0x015be8bad14c79d2722a0bd8bbe0cd93b905556d,-0.009430,-0.219729,348,240,299


## 3. Select the copy universe directly in the notebook

In [20]:
copy_mask = (
    (wallet_metrics['buy_roi'] >= COPY_MIN_BUY_ROI)
    & (wallet_metrics['num_buckets'] >= COPY_MIN_BUCKETS)
    & (wallet_metrics['num_markets'] >= COPY_MIN_MARKETS)
    & (wallet_metrics['trade_count'] >= COPY_MIN_TRADE_COUNT)
    & (wallet_metrics['max_drawdown_to_pnl'].fillna(1.0) <= COPY_MAX_DD_TO_PNL)
    & (wallet_metrics['copyable_roi'].fillna(0.0) >= COPY_MIN_COPYABLE_ROI)
)

copy_wallets = set(wallet_metrics.loc[copy_mask, 'wallet'])
copy_wallet_table = wallet_metrics.loc[copy_mask].copy()

print(f'copy_wallets={len(copy_wallets)}')
display(copy_wallet_table[['wallet', 'buy_roi', 'copyable_roi', 'trade_count', 'num_markets', 'num_buckets']].head(10))


copy_wallets=58


,wallet,buy_roi,copyable_roi,trade_count,num_markets,num_buckets
1,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0.039375,0.069611,210,180,208
39,0x07d601375c9bbb9037ad3c7a8f8fa0deff8164fb,0.083860,0.054305,491,89,353
52,0x0a59a7f2a870392a5f555aaa11f49d612e748e5c,0.037250,0.097951,1459,651,1106
59,0x0af29cc14fb231218c8d005685ac3db427a1ce86,0.088851,0.101813,1559,82,886
66,0x0d1048ec2a35bb690b217181bf5859d772b9a713,0.105692,0.055301,491,105,340
90,0x10484477985de3c6e9869f104f2d13b59a8ce362,0.141095,0.056424,517,155,476
123,0x177500541ae20bb0d46ab0db3fd2559e2a7e85b0,0.080500,0.059502,804,62,463
152,0x1c72101b1db0c7f4447edf999c6701bde226f5ce,0.042375,0.083309,305,81,194
158,0x1d4811eb053f27006701d9b852541c672e2bba3c,0.040698,0.192799,100,24,76
170,0x1e7220f4ed4e8f654a2743c961a43b675ae7bd5d,0.076560,0.126778,178,63,156


## 4. Build candidate copyable BUY trades and residualized ROI

In [21]:
candidate_trades = df_full[
    df_full['wallet'].isin(copy_wallets) & (df_full['side'] == 'BUY')
].copy()

c_train, c_val, c_test = split_data(candidate_trades, method='chronological')
fit = fit_roi_residualizer(c_train['copyable_roi'], c_train['price'])
for frame in (c_train, c_val, c_test):
    frame['roi_res'] = residualized_roi(frame['copyable_roi'], frame['price'], fit)

conditions = set(candidate_trades['condition_id'].unique())
print(f'candidate_trades={len(candidate_trades):,} conditions={len(conditions):,}')
display(c_train[['wallet', 'condition_id', 'outcome', 'price', 'copyable_pnl', 'copyable_roi', 'roi_res']].head())


Chronological split: train <= 2026-05-31T00:00:00Z, val <= 2026-06-27T00:00:00Z, test > 2026-06-27T00:00:00Z
Method: chronological  |  Unique end dates: 92  (train=36, val=27, test=29)

  Train:     46,338 trades  (7,717 markets)
  Val:       52,217 trades  (10,200 markets)
  Test:      42,247 trades  (8,990 markets)
  Total:    140,802 trades  (26,907 markets)
candidate_trades=140,802 conditions=26,907


,wallet,condition_id,outcome,price,copyable_pnl,copyable_roi,roi_res
25,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x02336b1d5f9d52c8998bd56a57f4969f233fd0d92b8c...,Yes,0.170,0.000000e+00,NaN,0.380996
26,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x037f7df5081789ebca056ad6e272452931c6a1375c2e...,No,0.929,1.261213e-16,0.076426,-0.198300
27,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x07e7c570f4d38fa8786947312850b4fc9af50860ddc0...,Yes,0.009,0.000000e+00,NaN,0.422127
28,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x0a4ed7975dab87ca68a9b4cd7fbe12589a657cc4b493...,No,0.790,1.680000e+00,0.265823,-0.059108
29,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x0a4ed7975dab87ca68a9b4cd7fbe12589a657cc4b493...,No,0.760,7.741932e-01,0.315789,-0.040447


## 5. Define the archetype sets and inspect the flipper cohort

In [22]:
signal_sets = archetype_sets(wallet_metrics, hold_metrics, min_trade_count=ARCH_MIN_TRADE_COUNT)

flipper_wallets = signal_sets['flipper']
flipper_table = wallet_metrics[wallet_metrics['wallet'].isin(flipper_wallets)].merge(
    hold_metrics[['wallet', 'median_hold_min', 'median_flip_min', 'n_round_trips', 'round_trip_rate']],
    on='wallet',
    how='left',
)

print(f'flipper_wallets={len(flipper_wallets)}')
display(flipper_table[['wallet', 'trade_count', 'buy_roi', 'copyable_roi', 'median_flip_min', 'n_round_trips']].head(10))


flipper_wallets=161


,wallet,trade_count,buy_roi,copyable_roi,median_flip_min,n_round_trips
0,0x0377e02b4ed7aa13fd9442e80a6362198489ae31,507,0.046816,0.000000,3.733333,32
1,0x058327e8fec1fdf218c4c2c20f3a30db18887dd4,362,0.012862,-0.011056,0.191667,135
2,0x09a85cfe4d51e0cde12daa4d255b3f789fb447c1,554,0.030790,-0.013971,5.850000,31
3,0x0a59a7f2a870392a5f555aaa11f49d612e748e5c,1459,0.037250,0.097951,15.750000,53
4,0x0af29cc14fb231218c8d005685ac3db427a1ce86,1559,0.088851,0.101813,16.500000,203
5,0x0e0cc261b9315683bbac3ebdd0a5be75fc29dde1,166,0.032844,0.152234,14.400000,24
6,0x0f37cb80dee49d55b5f6d9e595d52591d6371410,131,-0.017913,-0.173196,0.050000,25
7,0x10968af4773b31a1d97e1a4f072376de12cfa162,663,0.022484,0.000000,2.766667,72
8,0x10ab807ac93681f1f8d567501721ce245ec5b2e8,179,0.017900,-0.000000,15.666667,44
9,0x116db6298abcdefe06f9f5458c293c7de185fbf1,3564,0.007165,-0.000000,4.208333,208


## 6. Build the Signals using Strategies

We build a restricted trade frame (the position-checkpoint input) and use our modular `SignalStrategy` objects to attach the signals directly to the candidate splits.

In [23]:
restricted = df_full[df_full['condition_id'].isin(conditions)][ENGINE_COLS].copy()
candidate_splits = {"train": c_train, "val": c_val, "test": c_test}

# Instantiate our strategies
strategies = [
    GamblerCapitulationSqueeze(),
    FreshOppositeCrowdingFilter()
]

for strategy in strategies:
    print(f"Running strategy: {strategy.name}")
    candidate_splits = strategy.calculate_signals(
        candidate_splits,
        trades=restricted,
        wallet_metrics=wallet_metrics,
        hold_metrics=hold_metrics,
    )

# The splits have been updated (strategies return fresh copies)
c_train, c_val, c_test = candidate_splits["train"], candidate_splits["val"], candidate_splits["test"]

display(c_train.head(5))


Running strategy: GamblerCapitulationSqueeze
       - attaching base signals for set: gambler
       - attaching base signals for set: retail
       - attaching base signals for set: whale
Running strategy: FreshOppositeCrowdingFilter
       - attaching base signals for set: flipper
       - attaching base signals for set: both_sides
       - attaching base signals for set: overseller


,wallet,condition_id,token_id,dt,side,position,quantity,price,usdc_amount,final_value_usdc,...,sig_pos_own_overseller,sig_pos_opp_overseller,sig_pos_total_overseller,sig_val_own_overseller,sig_val_opp_overseller,sig_val_total_overseller,sig_avgc_own_overseller,sig_avgc_opp_overseller,sig_uwl_own_overseller,sig_uwl_opp_overseller
25,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x02336b1d5f9d52c8998bd56a57f4969f233fd0d92b8c...,3257508152209046936578279743585333210523600906...,2026-05-16 00:57:59+00:00,BUY,13.00,13.00,0.170,2.21000,13.00,...,1800.700171,938.396741,2739.096912,210.796639,828.737837,1039.534477,-0.311390,0.064027,-95.322390,49.868542
26,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x037f7df5081789ebca056ad6e272452931c6a1375c2e...,5090185675894667627638629469746111615262863706...,2026-05-17 01:53:56+00:00,BUY,13.00,13.00,0.929,12.07700,13.00,...,456.772381,158.337665,615.110046,443.936434,3.606860,447.543294,0.046177,-0.679161,19.594892,-7.635114
27,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x07e7c570f4d38fa8786947312850b4fc9af50860ddc0...,5038446518373887850102754415076629750222359315...,2026-05-17 03:12:39+00:00,BUY,2.95,2.95,0.009,0.02655,2.95,...,11.806423,75.600000,87.406423,0.105433,74.258110,74.363543,-0.007762,-0.008829,-0.000825,-0.661490
28,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x0a4ed7975dab87ca68a9b4cd7fbe12589a657cc4b493...,1869059801128472495359723481080178692749159122...,2026-05-18 01:24:45+00:00,BUY,8.00,8.00,0.790,6.32000,8.00,...,16.100000,38.340819,54.440819,10.430000,8.223005,18.653005,-0.179967,0.021292,-2.289000,0.171433
29,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x0a4ed7975dab87ca68a9b4cd7fbe12589a657cc4b493...,1869059801128472495359723481080178692749159122...,2026-05-18 01:28:26+00:00,BUY,13.00,5.00,0.760,3.80000,5.00,...,25.129181,31.870000,56.999181,17.036594,6.605300,23.641894,-0.107947,-0.136427,-2.061584,-1.043500


## 7. Turn it into a copy-trade signal and measure presence

In [24]:
for frame in (c_train, c_val, c_test):
    frame[COPY_SIGNAL_COL] = -frame[SIGNAL_COL].fillna(0.0)

presence_df = pd.DataFrame([
    {
        'split': label,
        'signal_gt_zero_share': float((frame[COPY_SIGNAL_COL] > 0).mean()),
        'signal_nonzero_share': float((frame[COPY_SIGNAL_COL] != 0).mean()),
        'mean_signal': float(frame[COPY_SIGNAL_COL].mean()),
        'IC_copyable_roi': compute_event_ic(frame[COPY_SIGNAL_COL], frame['copyable_roi']),
        'IC_roi_res': compute_event_ic(frame[COPY_SIGNAL_COL], frame['roi_res']),
    }
    for label, frame in [('train', c_train), ('val', c_val), ('test', c_test)]
])

pooled = pd.concat([c_train[[COPY_SIGNAL_COL, 'copyable_roi', 'roi_res']], c_val[[COPY_SIGNAL_COL, 'copyable_roi', 'roi_res']]], ignore_index=True)
boot_mean_roi, boot_lo_roi, boot_hi_roi = bootstrap_ic(pooled[COPY_SIGNAL_COL], pooled['copyable_roi'], n_iter=500, alpha=0.05, seed=42)
boot_mean_res, boot_lo_res, boot_hi_res = bootstrap_ic(pooled[COPY_SIGNAL_COL], pooled['roi_res'], n_iter=500, alpha=0.05, seed=42)

display(presence_df.round(4))
print(f'pooled train+val bootstrap IC on copyable_roi: mean={boot_mean_roi:+.4f} ci=({boot_lo_roi:+.4f}, {boot_hi_roi:+.4f})')
print(f'pooled train+val bootstrap IC on roi_res:      mean={boot_mean_res:+.4f} ci=({boot_lo_res:+.4f}, {boot_hi_res:+.4f})')


KeyError: 'sig_fval_opp_24h_both_sides'

## 8. Fit a train-only score transform and choose a threshold on validation

In [ ]:
score_fit = fit_rank_transformer(c_train[COPY_SIGNAL_COL].fillna(0.0))
for frame in (c_train, c_val, c_test):
    frame[SCORE_COL] = apply_rank_transformer(frame[COPY_SIGNAL_COL].fillna(0.0), score_fit)

val_grid = evaluate_threshold_grid(c_val, SCORE_COL, cost_bps=COST_BPS)
best_val = val_grid[val_grid['trades'] >= THRESHOLD_MIN_TRADES]
if best_val.empty:
    best_val = val_grid
best_row = best_val.sort_values('copyable_pnl_net', ascending=False).iloc[0]
best_threshold = float(best_row['threshold'])

display(val_grid.sort_values('copyable_pnl_net', ascending=False).head(10).round(4))
print(f'best_threshold={best_threshold:+.2f}')


/var/folders/j8/0dbnwk8n6m933m843h7hb88w0000gn/T/ipykernel_43835/161341288.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  frame[SCORE_COL] = apply_rank_transformer(frame[COPY_SIGNAL_COL].fillna(0.0), score_fit)
/var/folders/j8/0dbnwk8n6m933m843h7hb88w0000gn/T/ipykernel_43835/161341288.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  frame[SCORE_COL] = apply_rank_transformer(frame[COPY_SIGNAL_COL].fillna(0.0), score_fit)
/var/folders/j8/0dbnwk8n6m933m843h7hb88w0000gn/T/ipykernel_43835/161341288.py:3: PerformanceWarning: D

,threshold,trades,copyable_pnl,copyable_roi,copyable_pnl_net,copyable_roi_net,total_pnl,notional,copyable_notional,firing_rate,cost_paid,pnl_per_trade_net
6,-0.70,49195,4561.2002,0.0227,4561.2002,0.0227,32059.0445,911306.7878,200548.1046,0.9421,0.0,0.0927
7,-0.65,48608,4454.9713,0.0225,4454.9713,0.0225,31425.1676,901586.1131,198109.1400,0.9309,0.0,0.0917
5,-0.75,49741,4338.4388,0.0213,4338.4388,0.0213,32211.6673,926244.4664,203819.3752,0.9526,0.0,0.0872
8,-0.60,47861,3994.7633,0.0210,3994.7633,0.0210,30120.9099,877844.2541,189796.3743,0.9166,0.0,0.0835
9,-0.55,47244,3468.9029,0.0186,3468.9029,0.0186,28854.6343,864766.8741,186578.7611,0.9048,0.0,0.0734
4,-0.80,50329,3448.2398,0.0167,3448.2398,0.0167,30961.8524,935673.3451,207082.1606,0.9638,0.0,0.0685
2,-0.90,51150,3020.7497,0.0142,3020.7497,0.0142,29824.4625,950724.3706,213439.8397,0.9796,0.0,0.0591
10,-0.50,46510,2910.4596,0.0160,2910.4596,0.0160,27125.8689,854257.1709,182222.6204,0.8907,0.0,0.0626
11,-0.45,45704,2906.9408,0.0163,2906.9408,0.0163,26056.1877,839458.0476,178701.4559,0.8753,0.0,0.0636
1,-0.95,51596,2858.0378,0.0133,2858.0378,0.0133,30340.6477,953024.6281,214974.8873,0.9881,0.0,0.0554


best_threshold=-0.70


## 9. Findings: all candidate BUYs vs signal-filtered BUYs

This is the main before/after view.

- `all_candidates`: copy every candidate BUY in the copy universe
- `selected_by_signal`: copy only trades whose score clears the validation-chosen threshold

In [ ]:
def summarize_split(label, frame):
    baseline = evaluate_strategy(frame, SCORE_COL, -np.inf, cost_bps=COST_BPS)
    selected = evaluate_strategy(frame, SCORE_COL, best_threshold, cost_bps=COST_BPS)
    return [
        {
            'split': label,
            'group': 'all_candidates',
            'trades': baseline['trades'],
            'copyable_notional': baseline['copyable_notional'],
            'copyable_pnl_net': baseline['copyable_pnl_net'],
            'copyable_roi_net': baseline['copyable_roi_net'],
            'pnl_per_trade_net': baseline['copyable_pnl_net'] / max(baseline['trades'], 1),
            'firing_rate': baseline['firing_rate'],
        },
        {
            'split': label,
            'group': 'selected_by_signal',
            'trades': selected['trades'],
            'copyable_notional': selected['copyable_notional'],
            'copyable_pnl_net': selected['copyable_pnl_net'],
            'copyable_roi_net': selected['copyable_roi_net'],
            'pnl_per_trade_net': selected['copyable_pnl_net'] / max(selected['trades'], 1),
            'firing_rate': selected['firing_rate'],
        },
        {
            'split': label,
            'group': 'delta_selected_minus_all',
            'trades': selected['trades'] - baseline['trades'],
            'copyable_notional': selected['copyable_notional'] - baseline['copyable_notional'],
            'copyable_pnl_net': selected['copyable_pnl_net'] - baseline['copyable_pnl_net'],
            'copyable_roi_net': selected['copyable_roi_net'] - baseline['copyable_roi_net'],
            'pnl_per_trade_net': (selected['copyable_pnl_net'] / max(selected['trades'], 1)) - (baseline['copyable_pnl_net'] / max(baseline['trades'], 1)),
            'firing_rate': selected['firing_rate'] - baseline['firing_rate'],
        },
    ]

findings = pd.DataFrame(
    summarize_split('train', c_train)
    + summarize_split('val', c_val)
    + summarize_split('test', c_test)
)
display(findings.round(4))


,split,group,trades,copyable_notional,copyable_pnl_net,copyable_roi_net,pnl_per_trade_net,firing_rate
0,train,all_candidates,46338,274635.7344,29562.4908,0.1076,0.6380,1.0000
1,train,selected_by_signal,39387,213422.1610,20766.7960,0.0973,0.5273,0.8500
2,train,delta_selected_minus_all,-6951,-61213.5734,-8795.6948,-0.0103,-0.1107,-0.1500
3,val,all_candidates,52217,217015.6102,2070.1273,0.0095,0.0396,1.0000
4,val,selected_by_signal,49195,200548.1046,4561.2002,0.0227,0.0927,0.9421
5,val,delta_selected_minus_all,-3022,-16467.5055,2491.0729,0.0132,0.0531,-0.0579
6,test,all_candidates,42247,205129.2990,9605.3161,0.0468,0.2274,1.0000
7,test,selected_by_signal,39922,188676.8546,7556.7038,0.0401,0.1893,0.9450
8,test,delta_selected_minus_all,-2325,-16452.4444,-2048.6123,-0.0068,-0.0381,-0.0550


## 10. Trade-level examples

Inspect actual candidate trades with signal, score, and whether they would be copied after thresholding.

In [ ]:
test_view = c_test[[
    'dt', 'wallet', 'condition_id', 'outcome', 'price',
    'copyable_notional', 'copyable_pnl', 'copyable_roi', 'roi_res',
    SIGNAL_COL, COPY_SIGNAL_COL, SCORE_COL,
]].copy()
test_view['copy_trade'] = test_view[SCORE_COL] >= best_threshold

display(test_view.sort_values(SCORE_COL, ascending=False).head(2).round(4))
display(test_view[test_view['copy_trade']].sort_values('copyable_pnl', ascending=False).head(2).round(4))


,dt,wallet,condition_id,outcome,price,copyable_notional,copyable_pnl,copyable_roi,roi_res,sig_fval_opp_24h_both_sides,sig_copy_anti_crowding_flipper,score_flipper,copy_trade
12481001,2026-07-18 22:41:24+00:00,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0xe0ebca6e861b31be6f7dd328fd7d346cae5d0480bb28...,Yes,0.39,2.3769,-2.3769,-1.0,-1.0267,-0.0,0.0,1.0,True
1753096,2026-07-19 02:54:18+00:00,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x23e12be0bd7959375a225eccc0cba3da434d60322b01...,No,0.65,0.0000,0.0000,NaN,0.5440,-0.0,0.0,1.0,True


,dt,wallet,condition_id,outcome,price,copyable_notional,copyable_pnl,copyable_roi,roi_res,sig_fval_opp_24h_both_sides,sig_copy_anti_crowding_flipper,score_flipper,copy_trade
14011683,2026-07-14 04:55:17+00:00,0xc60bb547dbdf59c8d1746bc8a754accc3809d9f7,0xf2c9a5f2c7adf9cd8d7605d7616f5c5dbc3a048d8f54...,Yes,0.2329,86.5505,285.0406,3.2933,0.3557,408.0041,-408.0041,-0.4745,True
6867556,2026-06-30 05:26:49+00:00,0xc60bb547dbdf59c8d1746bc8a754accc3809d9f7,0x7ad3e06af5f5ead389da07df780468b4b14b0be8503d...,Yes,0.2200,57.6686,204.4614,3.5455,0.3595,50.0266,-50.0266,0.5062,True


## 11. Summary verdict

In [ ]:
test_baseline = findings[(findings['split'] == 'test') & (findings['group'] == 'all_candidates')].iloc[0]
test_selected = findings[(findings['split'] == 'test') & (findings['group'] == 'selected_by_signal')].iloc[0]
val_baseline = findings[(findings['split'] == 'val') & (findings['group'] == 'all_candidates')].iloc[0]
val_selected = findings[(findings['split'] == 'val') & (findings['group'] == 'selected_by_signal')].iloc[0]
test_ic_row = presence_df[presence_df['split'] == 'test'].iloc[0]
val_ic_row = presence_df[presence_df['split'] == 'val'].iloc[0]

summary = pd.DataFrame([
    {'metric': 'validation IC on copyable_roi', 'value': round(float(val_ic_row['IC_copyable_roi']), 4)},
    {'metric': 'test IC on copyable_roi', 'value': round(float(test_ic_row['IC_copyable_roi']), 4)},
    {'metric': 'validation IC on roi_res', 'value': round(float(val_ic_row['IC_roi_res']), 4)},
    {'metric': 'test IC on roi_res', 'value': round(float(test_ic_row['IC_roi_res']), 4)},
    {'metric': 'validation ROI improvement', 'value': round(float(val_selected['copyable_roi_net'] - val_baseline['copyable_roi_net']), 4)},
    {'metric': 'test ROI improvement', 'value': round(float(test_selected['copyable_roi_net'] - test_baseline['copyable_roi_net']), 4)},
    {'metric': 'validation firing rate', 'value': round(float(val_selected['firing_rate']), 4)},
    {'metric': 'test firing rate', 'value': round(float(test_selected['firing_rate']), 4)},
])
display(summary)


,metric,value
0,validation IC on copyable_roi,0.0413
1,test IC on copyable_roi,-0.0550
2,validation IC on roi_res,0.1947
3,test IC on roi_res,0.1276
4,validation ROI improvement,0.0132
5,test ROI improvement,-0.0068
6,validation firing rate,0.9421
7,test firing rate,0.9450


## 12. Combined composite evaluation (all confirmed families)

Run the remaining confirmed strategies on the shared candidate universe, then
combine the deduplicated families into equal / IC-weighted / shrinkage-Markowitz
composite scores (weights fit on **train** only) and check composite IC per split
against the single-signal ICs above. Thresholds are selected on **validation** and
reported on **test** (never tuned on test).

The composite is fit to **`copyable_pnl`** — the dollar PnL the copy strategy
actually earns — not `roi_res`. Because raw `copyable_roi` is ~50% correlated with
`price`, we also report price-controlled views (`pnl_res` = train-fit
price-residualized PnL, and within-price-bin IC) so the raw edge is not confused
with "buying cheap".


In [ ]:
# 12a. Attach the remaining confirmed strategies to the shared universe.
# (GamblerCapitulationSqueeze + FreshOppositeCrowdingFilter already attached above.)
from signal_lab.strategies import (
    CopyCrowdEntryTiming,
    FadeReactiveSellFlow,
    UwlOppContrarian,
)
from signal_lab.stage1 import build_composite_scores
from signal_lab.signal_lib import spearman_rho

extra_strategies = [
    CopyCrowdEntryTiming(),
    FadeReactiveSellFlow(),
    UwlOppContrarian(),
]

splits = {"train": c_train, "val": c_val, "test": c_test}
for strategy in extra_strategies:
    print(f"Running strategy: {strategy.name}")
    splits = strategy.calculate_signals(
        splits,
        trades=restricted,
        wallet_metrics=wallet_metrics,
        hold_metrics=hold_metrics,
    )
c_train, c_val, c_test = splits["train"], splits["val"], splits["test"]
print("done")

# 12b. Fit target = the dollar PnL the copy strategy actually earns.
target_col = "copyable_pnl"

# 12c. Composite candidate set: one representative per confirmed family, then
# greedily append near-orthogonal signals (curated corr<0.70, cap 8).
from signal_lab.evaluate_composite import select_curated

all_sig_cols = [c for c in c_train.columns if c.startswith("sig_")]
candidates = select_curated(splits, all_sig_cols, target_col=target_col, corr_threshold=0.70)[:8]
print("Composite candidates:", list(candidates))

# 12d. Price-confound columns (train-fit): pnl_res + price deciles.
from signal_lab.evaluate_composite import add_price_residualized_pnl, add_price_bins

add_price_residualized_pnl(splits, target_col=target_col, out_col="pnl_res")
add_price_bins(splits)

norm_splits, schemes, _ = build_composite_scores(
    splits, list(candidates), roi_col=target_col, weight_split="train", shrinkage=0.5
)

# 12e. Composite IC per split vs target AND price-controlled views.
comp_ic = pd.DataFrame({
    "split": split,
    "scheme": scheme,
    "IC_target": compute_event_ic(
        norm_splits[split][f"composite_{scheme}"], norm_splits[split][target_col]),
    "IC_pnl_res": compute_event_ic(
        norm_splits[split][f"composite_{scheme}"], norm_splits[split]["pnl_res"]),
    "spearman_price": spearman_rho(
        norm_splits[split][f"composite_{scheme}"], norm_splits[split]["price"]),
} for split in ("train", "val", "test") for scheme in schemes)
print("\nComposite IC per split (target=copyable_pnl):")
display(comp_ic.round(4))

# 12f. Within-price-bin IC: does the edge survive inside a fixed price decile?
from signal_lab.evaluate_composite import within_price_bin_ic

bin_rows = []
for scheme in schemes:
    sub = within_price_bin_ic(norm_splits, f"composite_{scheme}", target_col)
    sub.insert(0, "scheme", scheme)
    bin_rows.append(sub)
print("\nWithin-price-bin IC (mean Spearman inside train-fit price deciles):")
display(pd.concat(bin_rows, ignore_index=True).round(4))

# 12g. Threshold selection on validation by net PnL; report before/after on test.
for scheme in schemes:
    col = f"composite_{scheme}"
    val_grid = evaluate_threshold_grid(norm_splits["val"], col)
    n_val = len(norm_splits["val"])
    cand = val_grid[(val_grid['trades'] >= max(500, int(0.01 * n_val))) & (val_grid['firing_rate'] <= 0.95)]
    if cand.empty:
        print(f"composite_{scheme}: no val candidate rows in firing band")
        continue
    best = cand.sort_values('copyable_pnl_net', ascending=False).iloc[0]
    thr = float(best['threshold'])
    test_grid = evaluate_threshold_grid(norm_splits["test"], col)
    t_match = test_grid[np.isclose(test_grid['threshold'], thr, atol=1e-9)]
    t_row = t_match.iloc[0] if not t_match.empty else test_grid.iloc[0]
    all_row = test_grid.iloc[0]
    print(f"\n--- composite_{scheme} (val threshold {thr:+.2f}) ---")
    display(pd.DataFrame([
        {'split': 'val', 'group': 'selected', 'trades': int(best['trades']),
         'firing_rate': float(best['firing_rate']),
         'copyable_pnl': float(best['copyable_pnl_net']),
         'copyable_roi': float(best['copyable_roi_net'])},
        {'split': 'test', 'group': 'all_candidates', 'trades': int(all_row['trades']),
         'firing_rate': float(all_row['firing_rate']),
         'copyable_pnl': float(all_row['copyable_pnl_net']),
         'copyable_roi': float(all_row['copyable_roi_net'])},
        {'split': 'test', 'group': 'selected_by_signal', 'trades': int(t_row['trades']),
         'firing_rate': float(t_row['firing_rate']),
         'copyable_pnl': float(t_row['copyable_pnl_net']),
         'copyable_roi': float(t_row['copyable_roi_net'])},
    ]).round(4))


**Reading this composite section carefully.**

The composite is fit to **`copyable_pnl`** (dollar PnL), which roughly doubles the
raw IC vs the `roi_res`-fit version (equal 0.21 / 0.21 / 0.19; ic_weighted 0.31 /
0.35 / 0.33; shrinkage 0.33 / 0.37 / 0.35 on train/val/test). But the raw edge is
mostly the **"buy cheap" price component**, not a within-price signal:

- `spearman_price` of the composite is ~0.3–0.7 (equal ~0.5/0.35/0.31).
- Price-controlled views collapse: `IC_pnl_res` drops to ~0.07–0.15 and
  within-price-bin IC to ~0.02–0.09 (equal ≈ 0).

Also, PnL-maximizing threshold selection on validation degenerates to near-full
firing (equal fires ~95% of candidates), because more trades adds PnL in a
positive-PnL regime. Treat the **composite IC panel (and its price-confound
decomposition)** as the finding. The composite is a real but price-dominated
PnL-ranking signal; a cost-aware, capital-constrained live study is required
before any sizing.


## 13. Capital-constrained sizing backtest

The threshold tables above have no capital constraint, so "more trades = more PnL"
degenerates to near-full firing. A real strategy has a fixed budget, so here we copy a
**score-proportional share quantity** (`qty = scale * max(0, score) * copyable_qty`,
clipped to `copyable_qty`) under a global `$10k` budget. Capital is locked from the
trade's `dt` until market resolution (`end_date_iso`), forcing trades to compete.

We compare the **price-exposed** composite (fit on `copyable_pnl`) against the
**price-controlled** composite (fit on `pnl_res`). If the price component is tradable
alpha, the price-exposed variant should size to higher risk-adjusted PnL; if it was a
variance artifact, both should size about the same. Scale is selected on **validation**
(Sharpe of daily PnL) and reported on **test** (never tuned on test).


In [ ]:
# 13a. Capital-constrained sizing on the notebook universe.
# (norm_splits + pnl_res + price_dec come from cell 27; scores in [-1, 1].)
from signal_lab.sizing import capital_constrained_sim, select_scale, sizing_sharpe

BUDGET = 10_000.0
SCALE_GRID = np.arange(0.1, 3.01, 0.1)

# 13b. Price-controlled composite (weights fit on train against pnl_res).
controlled, _, _ = build_composite_scores(
    splits, list(candidates), roi_col="pnl_res", weight_split="train", shrinkage=0.5
)

# 13c. Sizing for each scheme x variant; scale picked on val, reported on val+test.
sizing_rows = []
for scheme in schemes:
    for label, nrm in (("price_exposed", norm_splits), ("price_controlled", controlled)):
        col = f"composite_{scheme}"
        best_scale, grid = select_scale(
            nrm["val"], col, BUDGET, SCALE_GRID, primary="sharpe_daily"
        )
        if grid.empty:
            print(f"sizing {scheme}/{label}: no taken trades on val")
            continue
        for split in ("val", "test"):
            res = capital_constrained_sim(nrm[split], col, BUDGET, best_scale)
            sizing_rows.append({
                "scheme": scheme, "variant": label, "split": split,
                "scale": best_scale, "trades": res["trades"],
                "net_pnl": round(res["net_pnl"], 2),
                "peak_used": round(res["peak_used"], 2),
                "pnl_per_peak": round(res["net_pnl"] / res["peak_used"], 4) if res["peak_used"] > 0 else 0.0,
                "sharpe_daily": round(sizing_sharpe(res["daily_pnl"], 365.0), 3),
            })
print("\nSizing results (budget=$10k, scale picked on val):")
display(pd.DataFrame(sizing_rows).round(4))


NameError: name 'build_composite_scores' is not defined